### Speechinteraction with BERT

Download model from https://huggingface.co/docs/transformers/model_doc/mobilebert

Data processing: https://huggingface.co/docs/transformers/tasks/token_classification

Explanations to metrics: https://www.sciencedirect.com/science/article/pii/S1574954124002516

Token classification: https://huggingface.co/docs/transformers/tasks/token_classification

In [ ]:
# install necessary libraries
%pip install datasets
%pip install tokenizers
%pip install transformers
%pip install wandb
%pip install seqeval
%pip install pandas
%pip install evaluate
%pip install accelerate>=0.26.0
%pip install transformers[torch]
%pip install nltk
%pip install spacy
%pip install onnx
%pip install onnxscript
%pip install sentencepiece
%pip install torch


%pip install tf-keras

In [2]:
import nltk
nltk.download('punkt')
from nltk.stem import PorterStemmer
nltk.download('averaged_perceptron_tagger')

ModuleNotFoundError: No module named 'nltk'

In [ ]:
import spacy
nlp = spacy.load("de_core_news_sm")

### WANDB 
Setup login & Projekt to inspect training values (Memory Usage, 
Framework to inspect weights and biases: https://wandb.ai/site

In [3]:
import os
import wandb
os.environ["WANDB_API_KEY"]="5b17f51ce17027d3a87c08f1a51dd680505de467"      # insert your API Key here
os.environ["WANDB_ENTITY"]="raffaelbecirevic-htl-spengergasse"       # insert user name
os.environ["WANDB_PROJECT"]="saner"      #insert project name
os.environ["WANDB_START_TIMEOUT"] = "180"
os.environ["WANDB_START_METHOD"] = "thread"


wandb.login(key="5b17f51ce17027d3a87c08f1a51dd680505de467")


WandbCoreNotAvailableError: File not found: C:\Users\raffa\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\wandb\bin\wandb-core. Please contact support at support@wandb.com. Your platform is: Windows-10-10.0.19045-SP0.

Added from above (muss man nicht runnen)

Ab hier runnen

In [ ]:
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer, AutoTokenizer, DataCollatorWithPadding
import torch
import torch.nn as nn
import pandas as pd
from datasets import Dataset, DatasetDict
#import wandb # added for wandb

# -----------------------------------------------------------------------------
# 1) Prepare your intent label list and dataset
# -----------------------------------------------------------------------------
intent_label_list = [
#"UNBEKANNT",
"NAME_BEKOMMEN",
"ALTER_BEKOMMEN",
"ADDRESSE_BEKOMMEN",
"GEBURTSDATUM_BEKOMMEN",
"SOZIALVERSICHERUNGSNUMMER_BEKOMMEN",
#"GESCHLECHT_BEKOMMEN",
"TELEFONNUMMER_BEKOMMEN",
#"JETZTIGER_KLIENT_BEKOMMEN",
#"NÄCHSTER_EINSATZ_KLIENT_BEKOMMEN",
#"NÄCHSTER_EINSATZ_UHRZEIT_BEKOMMEN",
#"LETZTER_EINSATZ_KLIENT_BEKOMMEN",
#"LETZTER_EINSATZ_UHRZEIT_BEKOMMEN",
#"JETZTIGER_EINSATZ_ENDE_BEKOMMEN",
#"EINSATZ_VON_KLIENT_BEKOMMEN",
#"ANZAHL_EINSAETZE_HEUTE_BEKOMMEN",
#"BESUCHTE_KLIENTEN_HEUTE_BEKOMMEN",
#"ANZAHL_EINSAETZE_HEUTE_UEBRIG_BEKOMMEN",
#"GEWICHT_BEKOMMEN",
#"PULS_BEKOMMEN",
#"BLUTDRUCK_BEKOMMEN",
#"KÖRPERTEMPERATUR_BEKOMMEN",
#"BLUTZUCKER_BEKOMMEN",
#"VERWANDTE_BEKOMMEN",
#"EMERGENCY_CONTACT_BEKOMMEN"
#"LETZTER_BETREUER_BEKOMMEN",
#"LETZTE_BETREUUNG_UHRZEIT_BEKOMMEN"
]
intent_label2id = {label: i for i, label in enumerate(intent_label_list)}
id2intent_label = {i: label for label, i in intent_label2id.items()}

# Read your CSV with "tokens","intent" columns
#df_intent = pd.read_csv(r"C:\Users\helle\Documents\IntentRecognizerModel2\SELECT_Statements_from01_with_intent_excel_compatible_no_unk.csv", sep=";")
#df_intent = pd.read_csv(r"C:\Users\helle\Documents\IntentRecognizerModel2\SELECT_Statements_from01_with_intent_excel_compatible_including03.csv", sep=";")
df_intent = pd.read_csv(r"C:\r Schule\Diplomprojekt\IntentRecognizerModel2\IntentRecognizerModel2\SELECT_Statements_mit_sozialversicherungsnummer.csv", sep=";")



# Convert to a Hugging Face Dataset
dataset_intent = Dataset.from_pandas(df_intent)

# Split into train/validation
dataset_intent = dataset_intent.train_test_split(test_size=0.10, seed=42)

# -----------------------------------------------------------------------------
# 2) Load your tokenizer and create a preprocessing function
# -----------------------------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained("bert_tokenizer_uncased_new_data")

def preprocess_intent(example):
    # Tokenize the sentence with padding
    enc = tokenizer(example["tokens"], truncation=True, padding="max_length", max_length=128)
    # Create numeric intent ID
    enc["labels_intent"] = intent_label2id[example["intent"]]
    return enc

dataset_intent = dataset_intent.map(preprocess_intent, batched=False)

# Remove unnecessary columns and set format
dataset_intent = dataset_intent.remove_columns(["tokens", "intent"])
dataset_intent.set_format("torch")
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
# Build a DatasetDict with "train" and "validation"
dataset_intent = DatasetDict({
    "train": dataset_intent["train"],
    "validation": dataset_intent["test"]
})
print(dataset_intent["validation"])

# -----------------------------------------------------------------------------
# 3) Define a model class that loads and freezes your existing NER model
#    then adds a new linear layer for intent classification
# -----------------------------------------------------------------------------
class NERPlusIntentModel(nn.Module):
    def __init__(self, ner_model_path, num_intent_labels):
        super().__init__()
        # Load the already-trained NER model
        self.ner_model = AutoModelForTokenClassification.from_pretrained(
            ner_model_path,
            output_attentions=False,
            output_hidden_states=False
        )
        
        # Freeze all NER parameters (preserving NER performance)
        for param in self.ner_model.parameters():
            param.requires_grad = False
        
        # Add a new intent classification head
        hidden_size = self.ner_model.config.hidden_size
        self.intent_classifier = nn.Linear(hidden_size, num_intent_labels)

    def forward(
        self,
        input_ids,
        attention_mask=None,
        token_type_ids=None,
        labels_intent=None
    ):
        # Run the underlying BERT encoder from the NER model
        outputs = self.ner_model.bert(
            input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )
        
        # Try to get pooler_output; if None, use the first token of last_hidden_state
        pooled_output = outputs.pooler_output
        if pooled_output is None:
            # outputs.last_hidden_state shape: [batch_size, seq_len, hidden_size]
            pooled_output = outputs.last_hidden_state[:, 0, :]  # Use the first token ([CLS])
        
        # Intent logits
        logits_intent = self.intent_classifier(pooled_output)

        # Compute intent loss
        loss = None
        if labels_intent is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits_intent, labels_intent)

        return {
            "loss": loss,
            "logits_intent": logits_intent,
        }

# -----------------------------------------------------------------------------
# 4) Instantiate the new model, specifying your existing NER model and intent labels
# -----------------------------------------------------------------------------
num_intent_labels = len(intent_label_list)

model_intent = NERPlusIntentModel(
    ner_model_path="bert_model_uncased_new_data",  # path to your trained NER model
    num_intent_labels=num_intent_labels
)

# -----------------------------------------------------------------------------
# 5) Train only the new intent head
# -----------------------------------------------------------------------------
optimizer = torch.optim.AdamW(model_intent.intent_classifier.parameters(), lr=1e-4, weight_decay=0.01  # ✅ Added
)


# added for wandb
# Initialize wandb
#wandb.init(
#    project="intent_recognition",  # Your project name
#    name="intent_model_run",       # Name of this specific run
#    config={
#        "learning_rate": 1e-4,
#        "epochs": 250,
#        "batch_size": 8,
#        "model_type": "NERPlusIntentModel",
#        "num_intent_labels": num_intent_labels,
#    }
#)

print(dataset_intent["validation"][0])

# Define training arguments
training_args = TrainingArguments(
    output_dir="intent_output",
    eval_strategy="epoch",
    num_train_epochs=100,
    weight_decay=0.01,  # ✅ Added
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_steps=20,
    save_strategy="no",
    learning_rate=1e-4,     # Already defined in optimizer, you could also add it here
    remove_unused_columns=False,
    report_to="none",  # <== das ist wichtig!
    fp16=True  # <== Mixed Precision für schnelleren & effizienteren GPU-Trainingslauf

)

# Define a metric function
from sklearn.metrics import classification_report, accuracy_score

import seqeval
import evaluate
import re
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)  # sequence classification returns a [batch_size, num_labels] array
    overall_accuracy = accuracy_score(labels, predictions)
    
    # Explicitly specify labels corresponding to all classes.
    all_class_indices = list(range(len(intent_label_list)))
    
    report = classification_report(
        labels,
        predictions,
        labels=all_class_indices,
        target_names=intent_label_list,
        output_dict=True,
        zero_division=0
    )
    
    # Flatten the report into a single dictionary.
    metrics = {"overall_accuracy": overall_accuracy}
    for label in intent_label_list:
        metrics[f"{label}_precision"] = report[label]["precision"]
        metrics[f"{label}_recall"] = report[label]["recall"]
        metrics[f"{label}_f1"] = report[label]["f1-score"]
        metrics[f"{label}_support"] = report[label]["support"]
    
    # Also add macro and weighted averages.
    metrics["macro_precision"] = report["macro avg"]["precision"]
    metrics["macro_recall"] = report["macro avg"]["recall"]
    metrics["macro_f1"] = report["macro avg"]["f1-score"]
    metrics["weighted_precision"] = report["weighted avg"]["precision"]
    metrics["weighted_recall"] = report["weighted avg"]["recall"]
    metrics["weighted_f1"] = report["weighted avg"]["f1-score"]
    
    return metrics


# Build a Trainer
trainer_intent = Trainer(
    model=model_intent,
    args=training_args,
    train_dataset=dataset_intent["train"],
    eval_dataset=dataset_intent["validation"],
    compute_metrics=compute_metrics,
    data_collator=data_collator
)

# Train
trainer_intent.train()

# Evaluate
eval_results = trainer_intent.evaluate()
print("Evaluation:", eval_results)

# (Optional) Save your intent model
torch.save(model_intent.state_dict(), "intent_model.pth")

tokenizer.save_pretrained("intent_model_tokenizer")

#added for wandb

# Log model artifacts to wandb
#model_artifact = wandb.Artifact('intent_model', type='model')
#model_artifact.add_file('intent_model.pth')
#wandb.log_artifact(model_artifact)

# Log tokenizer files to wandb
#tokenizer_artifact = wandb.Artifact('intent_model_tokenizer', type='tokenizer')
#tokenizer_artifact.add_dir('intent_model_tokenizer')
#wandb.log_artifact(tokenizer_artifact)

# End the wandb run
#wandb.finish()

Map: 100%|██████████| 86/86 [00:00<00:00, 2973.41 examples/s]


Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels_intent'],
    num_rows: 86
})
{'input_ids': tensor([  102,   359,  1382,   250,  1803, 15692,  6392,   244,   127, 15534,
          133,   103,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,  

Epoch,Training Loss,Validation Loss,Overall Accuracy,Name Bekommen Precision,Name Bekommen Recall,Name Bekommen F1,Name Bekommen Support,Alter Bekommen Precision,Alter Bekommen Recall,Alter Bekommen F1,Alter Bekommen Support,Addresse Bekommen Precision,Addresse Bekommen Recall,Addresse Bekommen F1,Addresse Bekommen Support,Geburtsdatum Bekommen Precision,Geburtsdatum Bekommen Recall,Geburtsdatum Bekommen F1,Geburtsdatum Bekommen Support,Sozialversicherungsnummer Bekommen Precision,Sozialversicherungsnummer Bekommen Recall,Sozialversicherungsnummer Bekommen F1,Sozialversicherungsnummer Bekommen Support,Telefonnummer Bekommen Precision,Telefonnummer Bekommen Recall,Telefonnummer Bekommen F1,Telefonnummer Bekommen Support,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1
1,1.758900,1.628384,0.395349,0.190476,0.400000,0.258065,10.000000,0.750000,0.300000,0.428571,20.000000,0.000000,0.000000,0.000000,12.000000,0.545455,0.600000,0.571429,20.000000,0.700000,0.466667,0.560000,15.000000,0.208333,0.555556,0.303030,9.000000,0.399044,0.387037,0.353516,0.467312,0.395349,0.391953
2,1.625800,1.504576,0.453488,0.263158,0.500000,0.344828,10.000000,0.888889,0.400000,0.551724,20.000000,0.000000,0.000000,0.000000,12.000000,0.500000,0.650000,0.565217,20.000000,0.888889,0.533333,0.666667,15.000000,0.227273,0.555556,0.322581,9.000000,0.461368,0.439815,0.408503,0.532420,0.453488,0.449888
3,1.485300,1.405319,0.488372,0.294118,0.500000,0.370370,10.000000,0.888889,0.400000,0.551724,20.000000,0.500000,0.083333,0.142857,12.000000,0.470588,0.800000,0.592593,20.000000,0.888889,0.533333,0.666667,15.000000,0.266667,0.444444,0.333333,9.000000,0.551525,0.460185,0.442924,0.603070,0.488372,0.480283
4,1.385700,1.335495,0.511628,0.277778,0.500000,0.357143,10.000000,0.909091,0.500000,0.645161,20.000000,1.000000,0.083333,0.153846,12.000000,0.500000,0.800000,0.615385,20.000000,1.000000,0.533333,0.695652,15.000000,0.250000,0.444444,0.320000,9.000000,0.656145,0.476852,0.464531,0.700112,0.511628,0.510968
5,1.338800,1.278312,0.569767,0.285714,0.400000,0.333333,10.000000,0.900000,0.450000,0.600000,20.000000,0.600000,0.250000,0.352941,12.000000,0.571429,0.800000,0.666667,20.000000,0.928571,0.866667,0.896552,15.000000,0.266667,0.444444,0.333333,9.000000,0.592063,0.535185,0.530471,0.649003,0.569767,0.573840
6,1.243700,1.234520,0.581395,0.307692,0.400000,0.347826,10.000000,0.909091,0.500000,0.645161,20.000000,0.666667,0.333333,0.444444,12.000000,0.666667,0.800000,0.727273,20.000000,1.000000,0.800000,0.888889,15.000000,0.200000,0.444444,0.275862,9.000000,0.625019,0.546296,0.554909,0.690606,0.581395,0.605539
7,1.276100,1.187957,0.604651,0.352941,0.600000,0.444444,10.000000,1.000000,0.450000,0.620690,20.000000,1.000000,0.166667,0.285714,12.000000,0.545455,0.900000,0.679245,20.000000,0.928571,0.866667,0.896552,15.000000,0.363636,0.444444,0.400000,9.000000,0.698434,0.571296,0.554441,0.739998,0.604651,0.592093
8,1.233600,1.147522,0.627907,0.428571,0.600000,0.500000,10.000000,1.000000,0.500000,0.666667,20.000000,0.800000,0.333333,0.470588,12.000000,0.571429,0.800000,0.666667,20.000000,0.933333,0.933333,0.933333,15.000000,0.285714,0.444444,0.347826,9.000000,0.669841,0.601852,0.597513,0.719601,0.627907,0.633072
9,1.282300,1.111688,0.627907,0.400000,0.600000,0.480000,10.000000,0.909091,0.500000,0.645161,20.000000,0.833333,0.416667,0.555556,12.000000,0.640000,0.800000,0.711111,20.000000,0.928571,0.866667,0.896552,15.000000,0.266667,0.444444,0.333333,9.000000,0.662944,0.604630,0.603619,0.712912,0.627907,0.640005
10,1.149100,1.078613,0.639535,0.416667,0.500000,0.454545,10.000000,1.000000,0.500000,0.666667,20.000000,0.666667,0.333333,0.444444,12.000000,0.607143,0.850000,0.708333,20.000000,0.882353,1.000000,0.937500,15.000000,0.307692,0.444444,0.363636,9.000000,0.646754,0.604630,0.595854,0.701326,0.639535,0.636209


Evaluation: {'eval_loss': 0.5896821022033691, 'eval_overall_accuracy': 0.8372093023255814, 'eval_NAME_BEKOMMEN_precision': 0.9, 'eval_NAME_BEKOMMEN_recall': 0.9, 'eval_NAME_BEKOMMEN_f1': 0.9, 'eval_NAME_BEKOMMEN_support': 10.0, 'eval_ALTER_BEKOMMEN_precision': 1.0, 'eval_ALTER_BEKOMMEN_recall': 0.65, 'eval_ALTER_BEKOMMEN_f1': 0.7878787878787878, 'eval_ALTER_BEKOMMEN_support': 20.0, 'eval_ADDRESSE_BEKOMMEN_precision': 0.8333333333333334, 'eval_ADDRESSE_BEKOMMEN_recall': 0.8333333333333334, 'eval_ADDRESSE_BEKOMMEN_f1': 0.8333333333333334, 'eval_ADDRESSE_BEKOMMEN_support': 12.0, 'eval_GEBURTSDATUM_BEKOMMEN_precision': 0.7407407407407407, 'eval_GEBURTSDATUM_BEKOMMEN_recall': 1.0, 'eval_GEBURTSDATUM_BEKOMMEN_f1': 0.851063829787234, 'eval_GEBURTSDATUM_BEKOMMEN_support': 20.0, 'eval_SOZIALVERSICHERUNGSNUMMER_BEKOMMEN_precision': 0.8333333333333334, 'eval_SOZIALVERSICHERUNGSNUMMER_BEKOMMEN_recall': 1.0, 'eval_SOZIALVERSICHERUNGSNUMMER_BEKOMMEN_f1': 0.9090909090909091, 'eval_SOZIALVERSICHERUNGS

('intent_model_tokenizer\\tokenizer_config.json',
 'intent_model_tokenizer\\special_tokens_map.json',
 'intent_model_tokenizer\\vocab.txt',
 'intent_model_tokenizer\\added_tokens.json',
 'intent_model_tokenizer\\tokenizer.json')

In [6]:
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118


Looking in indexes: https://download.pytorch.org/whl/cu118
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: C:\Users\raffa\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [8]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Keine GPU gefunden")


True
NVIDIA GeForce RTX 2070 with Max-Q Design


Das auch runnen

In [13]:
# Test input sentence
sentence = "Was Herr Meier ecard nummer"

# Check which device the model is on
device = next(model_intent.parameters()).device
print(f"Model is on device: {device}")

# Tokenize the sentence and move to correct device
encoded = tokenizer(sentence, return_tensors="pt")
encoded = {k: v.to(device) for k, v in encoded.items()}


# ---------------------------
# Test Intent Recognition
# ---------------------------
model_intent.eval()
with torch.no_grad():
    # Forward pass through your custom model to get intent logits
    out_intent = model_intent(
        input_ids=encoded["input_ids"],
        attention_mask=encoded["attention_mask"]
    )
intent_logits = out_intent["logits_intent"]
predicted_intent_id = intent_logits.argmax(dim=-1).item()
predicted_intent = intent_label_list[predicted_intent_id]
print("Predicted Intent:", predicted_intent)

# ---------------------------
# Test NER Functionality
# ---------------------------
with torch.no_grad():
    # Run the underlying NER model (frozen) for token-level predictions
    ner_outputs = model_intent.ner_model(**encoded)
ner_logits = ner_outputs.logits  # shape: [batch, seq_len, num_labels]
predicted_token_ids = ner_logits.argmax(dim=-1).squeeze().tolist()

# Retrieve token strings
tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"].squeeze().tolist())

# Convert token label IDs to label names (using the original NER model config)
predicted_ner_labels = [model_intent.ner_model.config.id2label.get(tid, "UNK") for tid in predicted_token_ids]

print("Tokens:", tokens)
print("NER Predictions:", predicted_ner_labels)

Model is on device: cuda:0
Predicted Intent: SOZIALVERSICHERUNGSNUMMER_BEKOMMEN
Tokens: ['[CLS]', 'was', 'herr', 'meier', 'ec', '##ard', 'nummer', '[SEP]']
NER Predictions: ['B-Entity', 'O', 'B-Person', 'I-Person', 'B-Entity', 'B-Entity', 'B-Entity', 'B-Entity']


from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer

# Step 1: Light Regularization
# Increase dropout rate in the model configuration (if possible)
model = AutoModelForTokenClassification.from_pretrained(
    'bert-base-german-cased',
    num_labels=12,
    id2label=id2label,
    label2id=label2id,
    # Applying light regularization by setting hidden_dropout_prob (if supported)
    hidden_dropout_prob=0.2  # Default is 0.1, let's increase to 0.2
)

# Increase weight decay for a slightly stronger penalty on large weights
training_args = TrainingArguments(
    output_dir="./fine_tune_bert_output",
    eval_strategy='epoch',
    learning_rate=4e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.02,  # Adjust from 0.01 to 0.02 for slightly stronger regularization
    logging_steps=10,
    report_to="wandb",
    run_name="ner_light_regularization",
    save_strategy='no'
)
from sklearn.model_selection import KFold
import numpy as np
from datasets import DatasetDict

# Assume `dataset` is your full dataset, split it manually for cross-validation
kf = KFold(n_splits=5)
split_datasets = []

# Tokenized dataset is assumed to be a Hugging Face Dataset
all_data = np.array(tokenized_dataset["train"])

# Perform 5-fold cross-validation
for fold, (train_idx, val_idx) in enumerate(kf.split(all_data)):
    # Create train and validation splits for the current fold
    train_split = all_data[train_idx]
    val_split = all_data[val_idx]

    # Convert to DatasetDict format
    split_dataset = DatasetDict({
        'train': train_split,
        'validation': val_split
    })

    # Train the model on this fold
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=split_dataset["train"],
        eval_dataset=split_dataset["validation"],
        data_collator=data_collator,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics
    )
    
    print(f"Training on fold {fold + 1}")
    trainer.train()
    results = trainer.evaluate()
    print(f"Results for fold {fold + 1}: {results}")

# This way, you get performance metrics for each fold and can average them for overall performance.


In [ ]:
wandb.finish()